<a href="https://colab.research.google.com/github/Sang-jun2/my_portfolio/blob/main/Anomaly%20detection/%ED%85%94%EB%A0%88%EA%B7%B8%EB%9E%A8%20%EB%A9%94%EC%84%B8%EC%A7%80/%ED%85%94%EB%A0%88%EA%B7%B8%EB%9E%A8_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''--------------------------------- 텔레그램 api ---------------------------------------'''
!pip install python-telegram-bot
!pip install nest_asyncio

import requests

BOT_TOKEN = 'MY TOKEN'
url = f'Telegram 봇 URL'
response = requests.get(url)

print(response.json())

# 소리데이터 텔레그램 테스트 데이터 만들기

import tensorflow as tf
from tensorflow.keras.models import load_model
from python_telegram_bot import Bot
import numpy as np
import pandas as pd
import os
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import matplotlib.pyplot as plt
from keras.models import Model
from keras.layers import Input, Dense
from tqdm import tqdm
import librosa.display
from scipy.signal import butter, lfilter
import random
import glob

# 음성 파일이 있는 폴더 경로 설정 (드라이브 마운트 경로 사용)
wav_directory = '/content/drive/MyDrive/예비 프로젝트/test'

# 폴더 내의 모든 .wav 파일 목록 가져오기
all_wav_files = glob.glob(os.path.join(wav_directory, '*.wav'))

# 무작위로 100개의 파일 선택 (파일 개수가 100개 이상일 경우에만)
random_wav_files = random.sample(all_wav_files, 100) if len(all_wav_files) >= 100 else all_wav_files

# 선택된 파일 목록 출력
print(f"무작위로 선택된 파일 개수: {len(random_wav_files)}")
for file in random_wav_files:
    print(file)

# test.csv 파일 불러오기
test_df = pd.read_csv('/content/drive/MyDrive/예비 프로젝트/test.csv')

# FAN_TYPE을 equipment_ID로 열 이름 변경
test_df.rename(columns={'FAN_TYPE': 'equipment_ID'}, inplace=True)

# equipment_name 열 추가하고 모든 값을 'fan'으로 설정
test_df['equipment_name'] = 'fan'

import librosa
import numpy as np
import scipy.signal as signal
from tqdm import tqdm

# 저주파 필터링 함수 정의
def apply_bandpass_filter(audio, sr, low_cut=50, high_cut=2000, order=5):
    nyquist = 0.5 * sr
    low = low_cut / nyquist
    high = (high_cut - 1) / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_audio = signal.lfilter(b, a, audio)
    return filtered_audio


# MFCC 추출 함수 정의
def extract_mfcc(y, sr=16000, n_mfcc=64):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.mean(mfccs, axis=1)  # 1차원 축소

# 각 파일에 대해 MFCC 추출 및 필터링 적용하여 test_df에 추가
mfcc_features = []
wav_directory = '/content/drive/MyDrive/예비 프로젝트'

for wav_file_path in tqdm(test_df['SAMPLE_PATH']):  # SAMPLE_PATH 열이 파일 경로를 포함한다고 가정
    full_wav_file_path = os.path.join(wav_directory, wav_file_path[2:])
    # 오디오 파일 로드
    y, sr = librosa.load(full_wav_file_path, sr=16000)

    # 저주파 필터링 적용
    filtered_y = apply_bandpass_filter(y, sr, low_cut=50, high_cut=2000)

    # MFCC 추출
    mfcc = extract_mfcc(filtered_y, sr, n_mfcc=64)

    # 추출된 MFCC 값을 리스트에 추가
    mfcc_features.append(mfcc)

# test_df에 MFCC 열 추가
test_df['MFCC'] = mfcc_features

from sklearn.preprocessing import MinMaxScaler
import numpy as np

# MinMaxScaler 초기화
scaler = MinMaxScaler()

# MFCC 열을 리스트로 변환하여 스케일링 준비
mfcc_features = np.array(test_df['MFCC'].tolist())

# MinMaxScaler를 사용하여 스케일링 적용
scaled_mfcc = scaler.fit_transform(mfcc_features)

# 스케일링된 결과를 새로운 열에 추가
test_df['MFCC_SCALED'] = scaled_mfcc.tolist()

test_df = test_df.drop(columns=['SAMPLE_PATH', 'MFCC'])

test_df.rename(columns={'equipment_ID': 'equipment_id'}, inplace=True)

fan_test_data = test_df

from telegram import Bot
import pandas as pd
import numpy as np
import asyncio
import nest_asyncio
from keras.models import load_model
import time
import keras.losses as losses
from tensorflow.keras.layers import LSTM

# 소리데이터 텔레그램 api 구현

# 텔레그램 봇 설정
BOT_TOKEN = 'MY TOKEN 직접 입력'
CHAT_ID = 'CHAT_ID 직업 입력'

# 텔레그램 봇 객체 생성
bot = Bot(token=BOT_TOKEN)

# 이벤트 루프 설정
nest_asyncio.apply()

# 위험 감지 메시지 전송 함수
async def send_alert_message(equipment_name, equipment_id, timestamp):
    message = f"경고! {equipment_name}의 {equipment_id}번 설비에서 이상 징후가 예측됩니다. 즉시 점검 부탁드립니다. [발견 시각: {timestamp}]"
    await bot.send_message(chat_id=CHAT_ID, text=message)

# 최근 알림 전송 시간을 저장하는 딕셔너리
last_alert_time = {}

# 알림 빈도를 제한하는 함수
async def process_anomaly_detection(predictions, threshold, equipment_name, equipment_id):
    global last_alert_time
    current_time = time.time()

    # 특정 장비의 마지막 알림 전송 시간을 확인하여 빈도 제한 (예: 60초)
    if equipment_id in last_alert_time:
        if current_time - last_alert_time[equipment_id] < 60:
            return  # 60초 이내라면 알림 전송 생략

    # 이상 감지 시 알림 전송
    anomalies = predictions > threshold
    if np.any(anomalies):
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        await send_alert_message(equipment_name, equipment_id, timestamp)

        # 알림 전송 시간 업데이트
        last_alert_time[equipment_id] = current_time

# 모델 로드 및 데이터 파일 평가 함수
def evaluate_sound_data():
    # 사용자 정의 객체 설정
    custom_objects = {'LSTM': LSTM, 'MeanSquaredError': losses.MeanSquaredError}

    # 소리 데이터에 대한 모델 로드
    sound_model = load_model('/content/drive/MyDrive/예비 프로젝트/텔레그램 모델/fan_id_6_model_dropout.keras', custom_objects=custom_objects)

    # CSV 파일에서 소리 데이터 읽기
    sound_data_df = fan_test_data.copy()  # 원본 데이터를 유지하기 위해 복사본 사용

    # 주기적으로 데이터를 읽어와 평가 (실시간처럼 동작하도록)
    for index in range(len(sound_data_df)):
        # 데이터 한 행씩 가져오기 (1D -> 3D 배열로 변환)
        equipment_id = sound_data_df.iloc[index]['equipment_id']  # 장비 ID 열

        # MFCC 값을 NumPy 배열로 변환 (리스트에서 변환 필요)
        mfcc_values = np.array(sound_data_df.iloc[index]['MFCC_SCALED'])

        # MFCC 값을 3D 배열로 변환 (모델 입력 형식에 맞춤)
        sound_data = mfcc_values.reshape(1, -1, 1)  # (batch_size, time_steps, features) 형식


        # 소리 데이터 평가
        sound_reconstructed = sound_model.predict(sound_data)
        sound_reconstruction_error = np.mean(np.abs(sound_data - sound_reconstructed), axis=(1, 2))
        # 재구성 오차 출력
        print(f"[Fan] Equipment ID: {equipment_id}, Reconstruction Error: {sound_reconstruction_error}")
        loop.run_until_complete(process_anomaly_detection(sound_reconstruction_error, threshold=0.17, equipment_name="fan", equipment_id=equipment_id))

        # 실시간 데이터를 모방하기 위해 대기 시간 설정 (예: 1초)
        time.sleep(1)


# 메인 함수
if __name__ == "__main__":
    # 현재 실행 중인 이벤트 루프 가져오기
    loop = asyncio.get_event_loop()
    evaluate_sound_data()


'''----------------------------------- 진동 데이터 텔레그램 api 이용 --------------------------------------'''

from google.colab import files
uploaded = files.upload()

!unzip 테스트데이터셋6부터30까지.zip

import pandas as pd
import numpy as np
import glob

# 모든 CSV 파일을 읽어오기 위해 파일 패턴을 사용합니다
csv_files = glob.glob('*.csv')

# 읽은 CSV 파일들을 저장할 리스트 생성
data_list = []

# CSV 파일을 순회하면서 각 파일을 데이터프레임으로 읽고 리스트에 저장
for file in csv_files:
    df = pd.read_csv(file)
    data_list.append(df.values)

# 리스트에 저장된 배열들의 shape 확인
for i, arr in enumerate(data_list):
    print(f"Shape of array {i}: {arr.shape}")

# 모든 배열의 shape을 동일하게 만들기 위한 padding 또는 truncating 작업 필요
# 여기서는 예시로, 모든 배열의 row 수를 최소 row 수로 맞추는 truncating 작업을 수행합니다.
min_rows = min(arr.shape[0] for arr in data_list)
data_list = [arr[:min_rows, :] for arr in data_list]

# 리스트에 저장된 배열을 3차원 배열로 변환
array_3d = np.array(data_list)

print(array_3d.shape)

model_vibration = array_3d

from keras.models import load_model
import keras.losses
model_path = '/content/진동 데이터 모델_1.h5'

# mae (mean absolute error)를 포함한 custom_objects 인수 전달
model = load_model(model_path, custom_objects={'mae': keras.losses.MeanAbsoluteError})

# 텔레그램 봇 설정
BOT_TOKEN = 'MY TOKEN 직접 입력'
CHAT_ID = 'CHAT_ID 직업 입력'

# 텔레그램 봇 객체 생성
bot = Bot(token=BOT_TOKEN)

# 이벤트 루프 설정
nest_asyncio.apply()

# equipment_id를 1에서 4 사이에서 랜덤하게 선택
random_equipment_id = random.randint(1, 4)

# 위험 감지 메시지 전송 함수
async def send_alert_message(equipment_name, equipment_id, timestamp):
    message = f"경고! bearing의 {random_equipment_id}번 설비에서 이상 징후가 예측됩니다. 즉시 점검 부탁드립니다. [발견 시각: {timestamp}]"
    await bot.send_message(chat_id=CHAT_ID, text=message)

# 최근 알림 전송 시간을 저장하는 딕셔너리
last_alert_time = {}

# 알림 빈도를 제한하는 함수
async def process_anomaly_detection(predictions, threshold, equipment_name, equipment_id):
    global last_alert_time
    current_time = time.time()

    # 특정 장비의 마지막 알림 전송 시간을 확인하여 빈도 제한 (예: 60초)
    if equipment_id in last_alert_time:
        if current_time - last_alert_time[equipment_id] < 60:
            return  # 60초 이내라면 알림 전송 생략

    # 이상 감지 시 알림 전송
    anomalies = predictions > threshold
    if np.any(anomalies):
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        await send_alert_message(equipment_name, equipment_id, timestamp)

        # 알림 전송 시간 업데이트
        last_alert_time[equipment_id] = current_time

# 모델 로드 및 데이터 파일 평가 함수
def evaluate_vibration_data():
    # 사용자 정의 객체 설정
    custom_objects = {'LSTM': LSTM, 'MeanSquaredError': losses.MeanSquaredError}

    # 진동 데이터에 대한 모델 로드
    vibration_model =  model_vibration

    # CSV 파일에서 진동 데이터 읽기
    vibration_data_df = array_3d.copy()  # 원본 데이터를 유지하기 위해 복사본 사용

    # 주기적으로 데이터를 읽어와 평가 (실시간처럼 동작하도록)
    for index in range(array_3d.shape[0]):
        # 장비 ID를 임의로 설정 (예: index를 장비 ID로 사용)
        equipment_id = index

        # 진동 데이터를 NumPy 배열로 가져옴 (3차원 배열의 각 slice 사용)
        vibration_values = array_3d[index]  # (time_steps, features)

        # 진동 값을 모델이 기대하는 (1, 1, 3) 형태로 변환
        time_steps = vibration_values.shape[0]

        # 모델이 예상하는 타임스텝 수와 맞추기 위해 슬라이싱 (여기서는 1로 고정)
        for i in range(time_steps):
            single_step_data = vibration_values[i:i+1, :]  # (1, features)
            single_step_data = single_step_data.reshape(1, 1, -1)  # (batch_size, time_steps, features)

            # 진동 데이터 평가
            vibration_reconstructed = vibration_model.predict(single_step_data)
            vibration_reconstruction_error = np.mean(np.abs(single_step_data - vibration_reconstructed), axis=(1, 2))
            # 재구성 오차 출력
            print(f"[Bearing] Equipment ID: {random_equipment_id}, Reconstruction Error: {vibration_reconstruction_error}")
            # 이상 감지
            loop.run_until_complete(process_anomaly_detection(vibration_reconstruction_error, threshold=0.02, equipment_name="vibration", equipment_id=equipment_id))

            # 실시간 데이터를 모방하기 위해 대기 시간 설정 (예: 1초)
            time.sleep(1)

# 메인 함수
if __name__ == "__main__":
    # 현재 실행 중인 이벤트 루프 가져오기
    loop = asyncio.get_event_loop()
    evaluate_vibration_data()


'''------------------------------------------- 결합 시도 ---------------------------------------'''
# 텔레그램 봇 설정
BOT_TOKEN = 'MY TOKEN 직접 입력'
CHAT_ID = 'CHAT_ID 직접 입력'

# 텔레그램 봇 객체 생성
bot = Bot(token=BOT_TOKEN)

# 이벤트 루프 설정
nest_asyncio.apply()

# 위험 감지 메시지 전송 함수
async def send_alert_message(equipment_name, equipment_id, timestamp):
    message = f"경고! {equipment_name}의 {equipment_id}번 설비에서 이상 징후가 예측됩니다. 즉시 점검 부탁드립니다. [발견 시각: {timestamp}]"
    await bot.send_message(chat_id=CHAT_ID, text=message)

# 최근 알림 전송 시간을 저장하는 딕셔너리
last_alert_time = {}

# 알림 빈도를 제한하는 함수
async def process_anomaly_detection(predictions, threshold, equipment_name, equipment_id):
    global last_alert_time
    current_time = time.time()

    # 특정 장비의 마지막 알림 전송 시간을 확인하여 빈도 제한 (예: 60초)
    if equipment_id in last_alert_time:
        if current_time - last_alert_time[equipment_id] < 60:
            return  # 60초 이내라면 알림 전송 생략

    # 이상 감지 시 알림 전송
    anomalies = predictions > threshold
    if np.any(anomalies):
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        await send_alert_message(equipment_name, equipment_id, timestamp)

        # 알림 전송 시간 업데이트
        last_alert_time[equipment_id] = current_time

# 소리 데이터 평가 함수
def evaluate_sound_data(sound_model, sound_data_df):
    for index in range(len(sound_data_df)):
        equipment_id = sound_data_df.iloc[index]['equipment_id']  # 장비 ID 열
        mfcc_values = np.array(sound_data_df.iloc[index]['MFCC_SCALED'])  # MFCC 값

        sound_data = mfcc_values.reshape(1, -1, 1)  # 3D 배열로 변환

        sound_reconstructed = sound_model.predict(sound_data)
        sound_reconstruction_error = np.mean(np.abs(sound_data - sound_reconstructed), axis=(1, 2))
        # 재구성 오차 출력
        print(f"[Sound] Equipment ID: {equipment_id}, Reconstruction Error: {sound_reconstruction_error}")

        # 이상 감지 함수 호출
        loop.run_until_complete(process_anomaly_detection(sound_reconstruction_error, threshold=0.15, equipment_name="소리", equipment_id=equipment_id))

        time.sleep(1)  # 실시간 데이터 처리 모방

# 진동 데이터 평가 함수
def evaluate_vibration_data(vibration_model, vibration_data_df):
    for index in range(vibration_data_df.shape[0]):
        equipment_id = index
        vibration_values = vibration_data_df[index]  # 진동 데이터
        time_steps = vibration_values.shape[0]

        for i in range(time_steps):
            single_step_data = vibration_values[i:i+1, :]
            single_step_data = single_step_data.reshape(1, 1, -1)  # 3D 배열로 변환

            vibration_reconstructed = vibration_model.predict(single_step_data)
            vibration_reconstruction_error = np.mean(np.abs(single_step_data - vibration_reconstructed), axis=(1, 2))
            # 재구성 오차 출력
            print(f"[Vibration] Equipment ID: {equipment_id}, Reconstruction Error: {vibration_reconstruction_error}")

            # 이상 감지 함수 호출
            loop.run_until_complete(process_anomaly_detection(vibration_reconstruction_error, threshold=0.2, equipment_name="진동", equipment_id=equipment_id))

            time.sleep(1)  # 실시간 데이터 처리 모방

# 메인 함수
if __name__ == "__main__":
    # 사용자 정의 객체 설정
    custom_objects = {'LSTM': LSTM, 'MeanSquaredError': losses.MeanSquaredError}

    # 소리 및 진동 모델 로드
    sound_model = load_model('/content/drive/MyDrive/예비 프로젝트/텔레그램 모델/fan_id_6_model_dropout.keras', custom_objects=custom_objects)
    vibration_model = model_vibration

    # 소리 및 진동 데이터 로드 (데이터프레임 또는 numpy 배열)
    sound_data_df = fan_test_data  # 소리 데이터
    vibration_data_df = model_vibration  # 진동 데이터

    # 현재 실행 중인 이벤트 루프 가져오기
    loop = asyncio.get_event_loop()

    # 소리 및 진동 데이터 평가 시작
    evaluate_sound_data(sound_model, sound_data_df)
    evaluate_vibration_data(vibration_model, vibration_data_df)